In [0]:
# Importing libraries

import os
import time
import datetime
import pandas as pd
from pyspark.sql.functions import col

os.environ['MLFLOW_DFS_TMP'] = '/Volumes/workspace/ml_layer/mlflow_tmp'

def ensure_catalog():
    spark.sql("USE CATALOG workspace")
    spark.sql("USE DATABASE ml_layer")

ensure_catalog()

# Pipeline configuration

PIPELINE_VERSION = "v1.0"
EXECUTION_DATE   = datetime.datetime.now().isoformat()

# Notebook execution path

BASE_PATH = "/Repos/jcharlieds05@gmail.com/AI-Risk-Fraud-Intelligence-Platform"

PIPELINE = [
    # DATA INGESTION (Bronze)
    {
        "stage": "1_bronze",
        "name": "Bronze BAF",
        "path": f"{BASE_PATH}/Data_Ingestion_&_Processing/Bronze Layer - Ingestion/bronze_layer_BAF",
        "timeout": 600,
        "critical": True
    },
    {
        "stage": "1_bronze",
        "name": "Bronze Fin Transactions",
        "path": f"{BASE_PATH}/Data_Ingestion_&_Processing/Bronze Layer - Ingestion/bronze_layer_fin_tran",
        "timeout": 600,
        "critical": True
    },
    {
        "stage": "1_bronze",
        "name": "Bronze Fraud Detection",
        "path": f"{BASE_PATH}/Data_Ingestion_&_Processing/Bronze Layer - Ingestion/bronze_layer_fraud_detection",
        "timeout": 600,
        "critical": True
    },
    {
        "stage": "1_bronze",
        "name": "Bronze PaySim",
        "path": f"{BASE_PATH}/Data_Ingestion_&_Processing/Bronze Layer - Ingestion/bronze_layer_paysim",
        "timeout": 600,
        "critical": True
    },

    # SILVER 
    {
        "stage": "2_silver",
        "name": "Silver BAF",
        "path": f"{BASE_PATH}/Data_Ingestion_&_Processing/Silver Layer - Transformation/silver_layer_BAF",
        "timeout": 600,
        "critical": True
    },
    {
        "stage": "2_silver",
        "name": "Silver Fraud Detection 2023",
        "path": f"{BASE_PATH}/Data_Ingestion_&_Processing/Silver Layer - Transformation/silver_layer_fraud_detection_2023",
        "timeout": 600,
        "critical": True
    },
    {
        "stage": "2_silver",
        "name": "Silver Fraud Detection 2025",
        "path": f"{BASE_PATH}/Data_Ingestion_&_Processing/Silver Layer - Transformation/silver_layer_fraud_detection_2025",
        "timeout": 600,
        "critical": True
    },
    {
        "stage": "2_silver",
        "name": "Silver PaySim",
        "path": f"{BASE_PATH}/Data_Ingestion_&_Processing/Silver Layer - Transformation/silver_layer_paysim",
        "timeout": 600,
        "critical": True
    },

    # GOLD 
    {
        "stage": "3_gold",
        "name": "Gold BAF",
        "path": f"{BASE_PATH}/Data_Ingestion_&_Processing/Gold Layer - Standarization/gold_layer_BAF",
        "timeout": 900,
        "critical": True
    },
    {
        "stage": "3_gold",
        "name": "Gold Fraud Detection 2023",
        "path": f"{BASE_PATH}/Data_Ingestion_&_Processing/Gold Layer - Standarization/gold_layer_fraud_detection_2023",
        "timeout": 900,
        "critical": True
    },
    {
        "stage": "3_gold",
        "name": "Gold Fraud Detection 2025",
        "path": f"{BASE_PATH}/Data_Ingestion_&_Processing/Gold Layer - Standarization/gold_layer_fraud_detection_2025",
        "timeout": 900,
        "critical": True
    },
    {
        "stage": "3_gold",
        "name": "Gold PaySim",
        "path": f"{BASE_PATH}/Data_Ingestion_&_Processing/Gold Layer - Standarization/gold_layer_paysim",
        "timeout": 900,
        "critical": True
    },
    {
        "stage": "3_gold",
        "name": "Gold Unified Fraud Transaction",
        "path": f"{BASE_PATH}/Data_Ingestion_&_Processing/Gold Layer - Standarization/gold_unified_fraud_transaction",
        "timeout": 1500,
        "critical": True
    },

    # ML PIPELINE
    {
        "stage": "5_features",
        "name": "Feature Engineering Applications",
        "path": f"{BASE_PATH}/ML_Pipeline/Feature_engineering_applications",
        "timeout": 900,
        "critical": True
    },
    {
        "stage": "5_features",
        "name": "Feature Engineering Transactions",
        "path": f"{BASE_PATH}/ML_Pipeline/Feature_engineering_transactions",
        "timeout": 1500,
        "critical": True
    },
    {
        "stage": "6_preprocessing",
        "name": "Null Imputation & Cleaning",
        "path": f"{BASE_PATH}/ML_Pipeline/null_imputation_cleaning",
        "timeout": 600,
        "critical": True
    },
    {
        "stage": "7_encoding",
        "name": "Encoding Applications",
        "path": f"{BASE_PATH}/ML_Pipeline/Encoding_applications",
        "timeout": 900,
        "critical": True
    },
    {
        "stage": "7_encoding",
        "name": "Encoding Transactions",
        "path": f"{BASE_PATH}/ML_Pipeline/Encoding_transactions",
        "timeout": 1500,
        "critical": True
    },

    # HYPERPARAMETER TUNING - Based on schedule
    {
        "stage": "8_optuna",
        "name": "Hyperparameter Tuning (Optuna)",
        "path": f"{BASE_PATH}/Modeling/08_hyperparameter_tuning",
        "timeout": 7200,
        "critical": False,
        "skip_default": True
    },

    #  MODEL TRAINING 
    {
        "stage": "9_training",
        "name": "Supervised Models",
        "path": f"{BASE_PATH}/Modeling/01_Supervised_models",
        "timeout": 3600,
        "critical": True
    },
    {
        "stage": "9_training",
        "name": "Anomaly Detection",
        "path": f"{BASE_PATH}/Modeling/02_Anomaly_Detection",
        "timeout": 3600,
        "critical": True
    },
    {
        "stage": "9_training",
        "name": "LSTM Transactions",
        "path": f"{BASE_PATH}/Modeling/03_lstm_transactions",
        "timeout": 9720,
        "critical": False
    },

    # EVALUATION & EXPLAINABILITY
    {
        "stage": "10_evaluation",
        "name": "Model Evaluation",
        "path": f"{BASE_PATH}/Modeling/04_model_evaluation",
        "timeout": 600,
        "critical": True
    },
    {
        "stage": "10_evaluation",
        "name": "Explainability (SHAP)",
        "path": f"{BASE_PATH}/Modeling/05_Explainability",
        "timeout": 1800,
        "critical": True
    },

    # PRODUCTION SCORING
    {
        "stage": "11_production",
        "name": "Combined Scoring & Business Impact",
        "path": f"{BASE_PATH}/Modeling/09_combined_scoring",
        "timeout": 1800,
        "critical": True
    },

    # MONITORING & DRIFT
    {
        "stage": "12_monitoring",
        "name": "Drift Detection & Retraining",
        "path": f"{BASE_PATH}/Modeling/06_monitoring",
        "timeout": 1200,
        "critical": True
    },

    # DASHBOARD REFRESH
    {
        "stage": "13_dashboard",
        "name": "Dashboard Data Prep",
        "path": f"{BASE_PATH}/Modeling/07_dashboard_data_prep",
        "timeout": 600,
        "critical": True
    }
]

# Widgets for control executions

dbutils.widgets.dropdown("execution_mode", "FULL", 
                         ["FULL", "ML_ONLY", "MODELING_ONLY", 
                          "MONITORING_ONLY", "WITH_OPTUNA"])
dbutils.widgets.dropdown("stop_on_failure", "true", ["true", "false"])

execution_mode  = dbutils.widgets.get("execution_mode")
stop_on_failure = dbutils.widgets.get("stop_on_failure") == "true"

# Determine which stages to run based on mode
STAGE_FILTERS = {
    "FULL"            : None,
    "ML_ONLY"         : ["5_features", "6_preprocessing", "7_encoding",
                         "9_training", "10_evaluation", "11_production",
                         "12_monitoring", "13_dashboard"],
    "MODELING_ONLY"   : ["9_training", "10_evaluation", "11_production",
                         "12_monitoring", "13_dashboard"],
    "MONITORING_ONLY" : ["10_evaluation", "11_production", "12_monitoring",
                         "13_dashboard"],
    "WITH_OPTUNA"     : None
}

stages_to_run = STAGE_FILTERS[execution_mode]
run_optuna    = (execution_mode == "WITH_OPTUNA")


# EXECUTION ENGINE

execution_log = []

def log_execution(notebook_name, stage, status, duration, error=None):
    """Track every notebook execution for audit trail."""
    execution_log.append({
        "execution_date" : EXECUTION_DATE,
        "pipeline_version": PIPELINE_VERSION,
        "stage"          : stage,
        "notebook"       : notebook_name,
        "status"         : status,
        "duration_sec"   : round(duration, 1),
        "error"          : str(error) if error else None,
        "timestamp"      : datetime.datetime.now().isoformat()
    })


def run_notebook(notebook_config):
    """Execute a single notebook with timeout and error handling."""
    name      = notebook_config["name"]
    path      = notebook_config["path"]
    timeout   = notebook_config["timeout"]
    critical  = notebook_config["critical"]
    stage     = notebook_config["stage"]

    print(f"\n{'-'*70}")
    print(f"  ▶ STAGE {stage} | {name}")
    print(f"  Path: {path}")
    print(f"{'-'*70}")

    start_time = time.time()
    try:
        result = dbutils.notebook.run(path, timeout)
        duration = time.time() - start_time
        log_execution(name, stage, "SUCCESS", duration)
        print(f"  ✅ Completed in {duration:.1f}s")
        return True

    except Exception as e:
        duration = time.time() - start_time
        log_execution(name, stage, "FAILED", duration, error=e)
        print(f"  ❌ FAILED after {duration:.1f}s")
        print(f"  Error details: {str(e)}")

        if critical and stop_on_failure:
            raise RuntimeError(f"Critical notebook '{name}' failed with error: {str(e)}")
        elif not critical:
            print(f"  ⚠️  Non-critical failure — continuing pipeline")

        return False


# Main Pipeline Execution

print("="*70)
print(f"  FRAUD INTELLIGENCE PLATFORM — PIPELINE EXECUTION")
print("="*70)
print(f"  Pipeline version : {PIPELINE_VERSION}")
print(f"  Execution mode   : {execution_mode}")
print(f"  Started at       : {EXECUTION_DATE}")
print(f"  Stop on failure  : {stop_on_failure}")
print("="*70)

pipeline_start = time.time()
total_succeeded, total_failed, total_skipped = 0, 0, 0

for nb in PIPELINE:
    # Apply stage filter
    if stages_to_run is not None and nb["stage"] not in stages_to_run:
        total_skipped += 1
        continue

    # Skip Optuna unless explicitly enabled
    if nb.get("skip_default", False) and not run_optuna:
        print(f"\n  ⏭️  Skipping {nb['name']} (use WITH_OPTUNA mode to include)")
        total_skipped += 1
        continue

    # Execute
    success = run_notebook(nb)
    if success:
        total_succeeded += 1
    else:
        total_failed += 1

# Execution Summary

pipeline_duration = time.time() - pipeline_start

print("\n" + "="*70)
print(f"  PIPELINE EXECUTION SUMMARY")
print("="*70)
print(f"  Total duration   : {pipeline_duration/60:.1f} minutes")
print(f"  ✅ Succeeded     : {total_succeeded}")
print(f"  ❌ Failed        : {total_failed}")
print(f"  ⏭️  Skipped       : {total_skipped}")
print("="*70)


# Persisting Execution Log to Delata for audit and monitoring dashboards

if execution_log:
    log_df = pd.DataFrame(execution_log)
    spark.createDataFrame(log_df).write.format("delta") \
        .mode("append") \
        .saveAsTable("workspace.ml_layer.pipeline_execution_log")

    print(f"\n  ✅ Execution log saved: {len(log_df)} entries")
    display(log_df)

# Exit with appropriate status
if total_failed > 0:
    raise RuntimeError(f"Pipeline completed with {total_failed} failure(s)")
else:
    print(f"\n  🎉 Pipeline completed successfully")